In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# VSM x Mechanism (variant-level non-binary data)
# Main Question: How important is a given mechanism/feature to a given VSM?
# Analysis:
## correlation of the given mechanism with a given VSM per ENSG
## Average coefficient of the given mechanism after fitting a linear regression model for a given VSM per ENSG (vsm_score ~ mechanism)
## VSM score and mechanism converted to within-protein rank

# feature list:
#### mutation rate, plddt, rsasa, foldx
#### new disorder metric
#### amino acid traits
#### AF2Bind pockets
#### G2P selected features (see other notebook)

In [3]:
chromosomes = [str(i) for i in range(1,23)] + ['X']
vsms = ['AM_score_percentile',
       'score_PAI3D_percentile', 'cadd_score_percentile',
       'cpt_score_percentile', 'gpn_msa_score_percentile',
       'MisFit_D_score_percentile', 'MisFit_S_score_percentile',
       'mpc_score_percentile', 'polyphen_score_percentile',
       'ESM_1v_percentile', 'EVE_percentile', 'popEVE_percentile',
       'mcap_score_percentile', 'esm1b_percentile', 'gmvp_score_percentile',
       'phylop_percentile', 'sift_score_percentile']
features = {'pLDDT': 'plddt', 'foldX': 'foldx_ddg', 'rSASA': 'rsasa', 'Mutation Rate': 'mu_snp',
           'MobiDB': 'mobidb_score'}
g2p_f_categorical = {'Secondary structure (DSSP 3-state)*': 
                     ['C (loop/coil)', 'H (helix)', 'B (undefined)']}
g2p_quantitative = {'Molar mass (g/mol)': 'molar_mass', 
                    'Hydropathy': 'hydropathy', 
                    'fpocket: druggability score*': 'druggability_score'}
g2p_f_binary = {'Compositional bias (UniProt)': 'compositional_bias', # would expect these variants to be benign
                'Cross-link (UniProt)': 'cross_link',
                'Disulfide bond (UniProt)': 'disulfide_bond', 
                'Domain (UniProt)': 'domain',
                'Glycosylation (UniProt)': 'glycosylation', 
                'Lipidation (UniProt)': 'lipidation', 
                'Motif (UniProt)': 'motif', 
                'Region (UniProt)': 'region',
                'Repeat (UniProt)': 'repeat',
                'Transmembrane (UniProt)': 'transmembrane',
                'Zinc finger (UniProt)': 'zinc_finger',
                'Acetylation': 'acetylation',
                'Methylation': 'methylation', 
                'Phosphorylation': 'phosphorylation', 
                'SUMOylation': 'SUMOylation', 
                'Ubiquitination': 'ubiquitination',
                'Intra-chain Hydrogen bond (AlphaFold2)': 'intra_chain_hydrogen_bond',
                'Intra-chain Non-bonded interaction (AlphaFold2)': 'intra_chain_nonbonded_interaction',
                'Intra-chain Salt bridge (AlphaFold2)': 'intra_chain_salt_bridge',
                'Signal (UniProt)': 'signal', 
                'Initiator methionine (UniProt)': 'initiator_methionine', 
                'Propeptide (UniProt)': 'propeptide', 
                'Peptide (UniProt)': 'peptide', 
                'Chain (UniProt)': 'chain', 
                'Topological domain (UniProt)': 'topological_domain', 
                'Intramembrane (UniProt)': 'intramembrane', 
                'Transit peptide (UniProt)': 'transit_peptide'}
g2p_change = {'Molar mass (g/mol)': 'delta_molar_mass', 
              'Hydropathy': 'delta_hydropathy'}
g2p_sites = ['Active site (UniProt)', 'Binding site (UniProt)', 'Site (UniProt)']

In [4]:
def load_and_merge_linker(df_c, chrom):
    df_l_c = pd.read_csv(
        f'gs://genetics-gym/linkers/linker_missense_enst_transcript_aa_uniprot_by_chrom_tsv/'
        f'linker_missense_enst_transcript_aa_uniprot_{chrom}.tsv.bgz',
        sep='\t',
        compression='gzip'
    )

    df_l_c = df_l_c[
        ['chrom', 'pos', 'ref', 'alt', 'ensg', 'enst',
         'uniprot_isoform', 'uniprot_id', 'aa_pos', 'aa_ref', 'aa_alt']
    ]

    df_cl = df_c.merge(
        df_l_c,
        how='inner',
        on=['chrom', 'pos', 'ref', 'alt', 'ensg']
    )

    if df_cl['aa_pos'].isna().any():
        print('warning: missing linker data. filtering out for now')
        df_cl = df_cl[df_cl.aa_pos.notna()].reset_index(drop=True)

    return df_cl

In [15]:
def merge_features(df_cl, df_f, column_list):
    return df_cl.merge(
        df_f,
        how='inner',
        on=column_list
    )

In [5]:
def add_within_gene_ranks(df, vsms, feature):
    for vsm in vsms:
        vsm_name = vsm.split('_percentile')[0]
        df[f'wgr_{vsm_name}'] = (
            df.groupby('ensg')[vsm]
              .rank(method='average', ascending=True)
        )

    df[f'wgr_{feature}'] = (
        df.groupby('ensg')[feature]
          .rank(method='average', ascending=True)
    )

    return df

In [6]:
def compute_gene_vsm_feature_corr(df_gene, vsms, feature, bool_feat):
    rows = []

    for vsm in vsms:
        vsm_name = vsm.split('_percentile')[0]
        x_col = f'wgr_{vsm_name}'
        if bool_feat:
            y_col=feature
        else:
            y_col = f'wgr_{feature}'

        df2 = df_gene[[x_col, y_col]].dropna()

        if len(df2) <= 1:
            return None

        X_raw = df2[[x_col]]
        if bool_feat:
            y_raw = df2[y_col].astype(bool)
        else:
            y_raw = df2[y_col]
        
        # ---- correlation ----
        corr = df2[x_col].corr(df2[y_col])

        rows.append({
            'vsm': vsm_name,
            'feature': feature,
            'corr': corr,
        })

    return rows

In [7]:
def compute_all_gene_results(df_clf, vsms, f):
    results = []

    genes = df_clf.ensg.unique()
    print(f'Genes to complete: {len(genes)}')

    for i, gene in enumerate(genes):
        df_gene = df_clf[df_clf.ensg == gene]

        rows = compute_gene_vsm_feature_corr(df_gene, vsms, f, False)

        if rows is not None:
            for r in rows:
                r['gene'] = gene
                results.append(r)

        # if i % 500 == 0:
        #     print(i)

    return pd.DataFrame(results)

In [8]:
def compute_all_gene_results_boolean(df_clf, vsms, f):
    results = []

    genes = df_clf.ensg.unique()
    print(f'Genes to complete: {len(genes)}')

    for i, gene in enumerate(genes):
        df_gene = df_clf[df_clf.ensg == gene]
        if len(df_gene[f].unique()) >1:
            rows = compute_gene_vsm_feature_corr(df_gene, vsms, f, True)
        else:
            rows = None

        if rows is not None:
            for r in rows:
                r['gene'] = gene
                results.append(r)

        # if i % 500 == 0:
        #     print(i)

    return pd.DataFrame(results)

In [9]:
def run_feature(df_cl, f, df_f, column_list, vsms):
    # add feature data onto variant/vsm data (inner join on column_list)
    df_clf = merge_features(df_cl, df_f, column_list)
    
    df_clf = add_within_gene_ranks(df_clf, vsms, f)
    results_df = compute_all_gene_results(df_clf, vsms, f)
    
    results_df.to_csv(
        f'all_stats/stats_vsm_{f}_genes_chr{chrom}.tsv',
        sep='\t',
        index=False
    )

In [10]:
def run_feature_boolean(df_clf, f, vsms):
    # merging data beforehand ***
    # # add feature data onto variant/vsm data (inner join on column_list)
    # df_clf = merge_features(df_cl, df_f, column_list)
    
    #df_clf = add_within_gene_ranks(df_clf, vsms, f)
    # only want within_gene_rank for vsms, not for boolean feature
    for vsm in vsms:
        vsm_name = vsm.split('_percentile')[0]
        df_clf[f'wgr_{vsm_name}'] = (
            df_clf.groupby('ensg')[vsm]
            .rank(method='average', ascending=True)
        )
    results_df = compute_all_gene_results_boolean(df_clf, vsms, f)
    
    results_df.to_csv(
        f'all_stats/stats_vsm_{f}_genes_chr{chrom}.tsv',
        sep='\t',
        index=False
    )

In [11]:
# feature data
# isoform information
df_p = pd.read_csv('../reference_data/protein_sequence_guide.tsv', sep='\t')
df_p = df_p[df_p.uniprot_canonical]
df_p = df_p[['uniprot_id', 'uniprot_isoform']].drop_duplicates()
# pLDDT
df_plddt = pd.read_csv('../reference_data/plddt_missense_variants_gr38.tsv.gz', sep='\t', compression='gzip')
# add in isoform information
df_plddt = df_plddt.merge(df_p, how='inner', on='uniprot_id')
df_plddt = df_plddt[['aa_pos', 'aa_ref', 'avg_plddt', 'uniprot_isoform']]
df_plddt = df_plddt.rename(columns={'avg_plddt': 'plddt'})
# mutation rate
df_mu = pd.read_csv('../reference_data/mutation_rate.tsv.bgz', sep='\t', compression='gzip')
# foldX
df_foldx = pd.read_csv('../reference_data/foldx/foldx_max_plddt_clean.tsv.gz', sep='\t', compression='gzip')
# add in isoform information
df_foldx = df_foldx.merge(df_p, how='inner', on='uniprot_id')
df_foldx = df_foldx[['aa_pos', 'aa_ref', 'aa_alt', 'foldx_ddg', 'uniprot_isoform']]
# rSASA
df_rsasa = pd.read_csv('../reference_data/rsasa_missense_variants_gr38.tsv.gz', sep='\t', compression='gzip')
# add in isoform information
df_rsasa = df_rsasa.merge(df_p, how='inner', on='uniprot_id')
df_rsasa = df_rsasa[['aa_pos', 'aa_ref', 'rsasa', 'uniprot_isoform']]

# af2bind
af2bind = pd.read_csv('../reference_data/af2bind/expanded_pockets.tsv', sep='\t')
# get individual residues in pockets
af2bind=af2bind[['uniprot', 'expanded_pocket']]
def parse_int_list(s):
    # remove brackets and newlines, then split on whitespace
    return list(map(int, s.strip('[]').replace('\n', ' ').split()))

af2bind['expanded_pocket'] = af2bind['expanded_pocket'].apply(parse_int_list)
af2bind_vars = af2bind.explode('expanded_pocket')
af2bind_vars = af2bind_vars.drop_duplicates()
af2bind_vars = af2bind_vars.rename(columns={'uniprot': 'uniprot_id', 'expanded_pocket': 'aa_pos'})
af2bind_vars['af2bind_pocket'] = True
# af2bind only gives ID not isoform - but should be based off of the canonical isoform?
af2bind_vars = af2bind_vars.merge(df_p, how='inner', on=['uniprot_id']) # gives us the canonical isoform (to the best of my ability)
af2bind_vars = af2bind_vars[["aa_pos", "af2bind_pocket", "uniprot_isoform"]]

# AA properties
aa_property = {'A': 'Nonpolar/Neutral', 'C': 'Polar/Neutral', 'D': 'Charged', 'E': 'Charged', 'F': 'Aromatic', 'G': 'Nonpolar/Neutral', 
           'H': 'Charged', 'I': 'Nonpolar/Neutral', 'K': 'Charged', 'L': 'Nonpolar/Neutral', 'M': 'Nonpolar/Neutral', 'N': 'Polar/Neutral', 
           'P': 'Nonpolar/Neutral', 'Q': 'Polar/Neutral', 'R': 'Charged', 'S': 'Polar/Neutral', 'T': 'Polar/Neutral', 
           'V': 'Nonpolar/Neutral', 'W': 'Aromatic', 'Y': 'Aromatic'}
#df_l['aa_property'] = df_l['aa_ref'].map(aa_property)

# alt disorder metric - mobidb
df_mobidb = pd.read_csv('../reference_data/mobidb/mobidb_disorder_per_residue.tsv.gz', sep='\t', compression='gzip')

# identity information of hydropathy and molar mass change per amino acid ref-alt pair
h_mm_df = pd.read_csv('../reference_data/hydrop_mm_guide.tsv', sep='\t')
h_mm_df = h_mm_df[['aa_ref', 'aa_alt', 'delta_hydropathy', 'delta_molar_mass']]

In [12]:
# vsm data - with ensg and on inner join
df = pd.read_parquet('gs://grohlicek/genetics_gym_vsm_all_content/vsm_all_2026_04_21/vsm_ensg_inner_pre.parquet')
# drop evaluation data columns - just want variant, ensg, and scores
df = df[['chrom', 'pos', 'ref', 'alt', 'AM_score_percentile',
       'score_PAI3D_percentile', 'cadd_score_percentile',
       'cpt_score_percentile', 'gpn_msa_score_percentile',
       'MisFit_D_score_percentile', 'MisFit_S_score_percentile',
       'mpc_score_percentile', 'polyphen_score_percentile',
       'ESM_1v_percentile', 'EVE_percentile', 'popEVE_percentile',
       'mcap_score_percentile', 'esm1b_percentile', 'gmvp_score_percentile',
       'phylop_percentile', 'sift_score_percentile', 'ensg']]
# data choices to debate
## remove where ensg is na
df = df[df.ensg.notna()]
## remove duplicates
df = df.drop_duplicates()
## clean up index
df = df.reset_index(drop=True)

In [13]:
# results directory
os.makedirs('all_stats', exist_ok=True)

In [16]:
for chrom in chromosomes:
    print(f'Chromosome {chrom}')

    df_c = df[df.chrom == 'chr' + chrom]

    df_cl = load_and_merge_linker(df_c, chrom)
    
    # run each feature

    print('starting features: foldx, mu, rsasa, plddt, mobidb, af2bind')
    # foldx
    f = 'foldx_ddg'
    #print(f'starting feature: {f}')
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref', 'aa_alt']
    run_feature(df_cl, f, df_foldx, join_list, vsms)
    
    # mutation rate
    f = 'mu_snp'
    #print(f'starting feature: {f}')
    join_list = ['chrom', 'pos', 'ref', 'alt']
    run_feature(df_cl, f, df_mu, join_list, vsms)
    
    # rsasa
    f = 'rsasa'
    #print(f'starting feature: {f}')
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref']
    run_feature(df_cl, f, df_rsasa, join_list, vsms)

    # plddt
    f = 'plddt'
    #print(f'starting feature: {f}')
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref']
    run_feature(df_cl, f, df_plddt, join_list, vsms)

    # mobidb
    f = 'mobidb_score'
    #print(f'starting feature: {f}')
    join_list = ['uniprot_isoform', 'aa_pos']
    run_feature(df_cl, f, df_mobidb, join_list, vsms)

    # af2bind
    f = 'af2bind_pocket'
    #print(f'starting feature: {f}')
    # list of valid uniprot ids
    uni_iso = af2bind_vars.uniprot_isoform.unique()
    # restrict data to variants on isoforms in af2bind dataset
    df_clf = df_cl[df_cl.uniprot_isoform.isin(uni_iso)]
    # now merge with af2bind pockets
    df_clf = df_clf.merge(
            af2bind_vars,
            how='left',
            on=['uniprot_isoform', 'aa_pos']
        )
    # replace NA with False
    df_clf['af2bind_pocket'] = df_clf.af2bind_pocket.fillna(False)
    run_feature_boolean(df_clf, f, vsms)

    # amino acid properties
    df_clf = df_cl.copy()
    df_clf['aa_property'] = df_clf['aa_ref'].map(aa_property)
    # one property at a time
    print('starting amino acid properties')
    for aap in df_clf.aa_property.unique():
        #print(f'starting feature: {aap}')
        f = aap.replace('/', '_')
        df_clf[f] = df_clf.aa_property==aap
        run_feature_boolean(df_clf, f, vsms)

    # g2p_change
    join_list = ['aa_ref', 'aa_alt']
    print('starting g2p change in hydropathy and molar mass')
    run_feature(df_cl, 'delta_hydropathy', h_mm_df, join_list, vsms)
    run_feature(df_cl, 'delta_molar_mass', h_mm_df, join_list, vsms)

Chromosome 1


/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_26621/4141627806.py:2: DtypeWarning: Columns (0: uniprot_isoform, 1: uniprot_id, 2: transcript_mane_select) have mixed types. Specify dtype option on import or set low_memory=False.
  df_l_c = pd.read_csv(


starting features: foldx, mu, rsasa, plddt, mobidb, af2bind
Genes to complete: 1466


/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Genes to complete: 1510


...

Genes to complete: 142
Genes to complete: 136
Genes to complete: 136
Genes to complete: 136
Genes to complete: 77
starting amino acid properties
Genes to complete: 142
Genes to complete: 142
Genes to complete: 142
Genes to complete: 142
starting g2p change in hydropathy and molar mass
Genes to complete: 142
Genes to complete: 142
Chromosome 22


/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_26621/4141627806.py:2: DtypeWarning: Columns (0: transcript_mane_select) have mixed types. Specify dtype option on import or set low_memory=False.
  df_l_c = pd.read_csv(


starting features: foldx, mu, rsasa, plddt, mobidb, af2bind
Genes to complete: 308
Genes to complete: 322
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 203
starting amino acid properties
Genes to complete: 322
Genes to complete: 322
Genes to complete: 322
Genes to complete: 322
starting g2p change in hydropathy and molar mass
Genes to complete: 322
Genes to complete: 322
Chromosome X


/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_26621/4141627806.py:2: DtypeWarning: Columns (0: transcript_mane_select) have mixed types. Specify dtype option on import or set low_memory=False.
  df_l_c = pd.read_csv(


starting features: foldx, mu, rsasa, plddt, mobidb, af2bind
Genes to complete: 480
Genes to complete: 491
Genes to complete: 480
Genes to complete: 480
Genes to complete: 481
Genes to complete: 300
starting amino acid properties
Genes to complete: 491
Genes to complete: 491
Genes to complete: 491
Genes to complete: 491
starting g2p change in hydropathy and molar mass
Genes to complete: 491
Genes to complete: 491


In [17]:
# g2p features
df_g2p = pd.read_csv('../reference_data/g2p_data_final_2.tsv.gz', sep='\t', compression='gzip')
# drop columns we no longer care about
df_g2p = df_g2p[['uniprot_id', 'aa_pos', 'AA']+list(g2p_f_categorical.keys())+list(g2p_f_binary.keys())+list(g2p_quantitative.keys())+g2p_sites]
# drop nonstandard amino acid U
df_g2p = df_g2p[df_g2p.AA!='U'].reset_index(drop=True)
# g2p only gives ID not isoform - but should be based off of the canonical isoform
df_g2p = df_g2p.merge(df_p, how='inner', on=['uniprot_id']) # gives us the canonical isoform (to the best of my ability)
# fix AA column name
df_g2p = df_g2p.rename(columns={'AA': 'aa_ref'})

/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_26621/1658042735.py:2: DtypeWarning: Columns (0: Secondary structure (PDBe/SIFTS), 1: Secondary structure (DSSP 3-state)*, 2: Secondary structure (DSSP 9-state)*, 3: Active site (UniProt), 4: Binding site (UniProt), 5: Coiled coil (UniProt), 6: Cross-link (UniProt), 7: Disulfide bond (UniProt), 8: DNA binding (UniProt), 9: Domain (UniProt), 10: Glycosylation (UniProt), 11: Initiator methionine (UniProt), 12: Intramembrane (UniProt), 13: Lipidation (UniProt), 14: Modified residue (UniProt), 15: Motif (UniProt), 16: Mutagenesis (UniProt), 17: Non-standard residue (UniProt), 18: Non-terminal residue (UniProt), 19: Peptide (UniProt), 20: Propeptide (UniProt), 21: Repeat (UniProt), 22: Signal (UniProt), 23: Site (UniProt), 24: Topological domain (UniProt), 25: Transit peptide (UniProt), 26: Transmembrane (UniProt), 27: Zinc finger (UniProt), 28: Acetylation, 29: Disease-associated PTMs, 30: Methylation, 31: O-GalNAc, 32: O-GlcNAc, 3

In [18]:
# G2P data
for chrom in chromosomes:
    print(f'Chromosome {chrom}')

    df_c = df[df.chrom == 'chr' + chrom]

    df_cl = load_and_merge_linker(df_c, chrom)
    
    # run each feature

    # G2P features - Quantitative
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref']
    print('starting quantitative g2p features')
    for f in g2p_quantitative.keys():
        #print(f'starting feature: {f}')
        df_g2p_f = df_g2p[join_list+[f]]
        df_g2p_f = df_g2p_f[df_g2p_f[f].notna()].reset_index(drop=True)
        df_g2p_f = df_g2p_f.rename(columns={f: g2p_quantitative[f]})
        if g2p_quantitative[f] == 'druggability_score': # this feature needs specific cleaning
            df_g2p_f[g2p_quantitative[f]] = df_g2p_f[g2p_quantitative[f]].astype(str)
            df_g2p_f[g2p_quantitative[f]] = df_g2p_f[g2p_quantitative[f]].str.split(': ').str[-1]
            df_g2p_f[g2p_quantitative[f]] = df_g2p_f[g2p_quantitative[f]].astype(float)
        run_feature(df_cl, g2p_quantitative[f], df_g2p_f, join_list, vsms)

    # G2P features - binary
    print('starting g2p binary features')
    for f in g2p_f_binary.keys():
        #print(f'starting feature: {f}')
        df_g2p_f = df_g2p[join_list+[f]]
        df_g2p_f[f] = df_g2p_f[f].notna()
        df_g2p_f = df_g2p_f.rename(columns={f: g2p_f_binary[f]})
        # list of valid uniprot ids
        uni_iso = df_g2p_f.uniprot_isoform.unique()
        # restrict data to variants on isoforms in g2p dataset
        df_clf = df_cl[df_cl.uniprot_isoform.isin(uni_iso)]
        # now merge with g2p data
        df_clf = df_clf.merge(
                df_g2p_f,
                how='left',
                on=['uniprot_isoform', 'aa_pos', 'aa_ref']
            )
        run_feature_boolean(df_clf, g2p_f_binary[f], vsms)

    # G2P features - categorical - just secondary structure
    f = 'Secondary structure (DSSP 3-state)*'
    df_g2p_f = df_g2p[join_list+[f]]
    # list of valid uniprot ids
    uni_iso = df_g2p_f.uniprot_isoform.unique()
    # restrict data to variants on isoforms in g2p dataset
    df_clf = df_cl[df_cl.uniprot_isoform.isin(uni_iso)]
    # now merge with af2bind pockets
    df_clf = df_clf.merge(
            df_g2p_f,
            how='left',
            on=['uniprot_isoform', 'aa_pos', 'aa_ref']
        )
    # one ss at a time
    ss_3 = {'C (loop/coil)': 'ss_loop_coil', 'H (helix)': 'ss_helix', 'B (undefined)': 'ss_undefined'}
    print('starting secondary structure features')
    for ss in ss_3:
        #print(f'starting feature: {ss}')
        df_clf[ss_3[ss]] = df_clf['Secondary structure (DSSP 3-state)*']==ss
        run_feature_boolean(df_clf, ss_3[ss], vsms)

    # g2p_sites - combine binding sites, active sites, and other sites
    print(f'starting feature: G2P all sites')
    join_list = ['uniprot_isoform', 'aa_pos', 'aa_ref']
    df_g2p_f = df_g2p[join_list+g2p_sites]
    #df_g2p_f = df_g2p_f[df_g2p_f[f].notna()].reset_index(drop=True)
    df_g2p_f['all_uniprot_sites'] = ((df_g2p_f[g2p_sites[0]].notna()) | (df_g2p_f[g2p_sites[1]].notna()) | (df_g2p_f[g2p_sites[2]].notna()))
    # list of valid uniprot ids
    uni_iso = df_g2p_f.uniprot_isoform.unique()
    # restrict data to variants on isoforms in g2p dataset
    df_clf = df_cl[df_cl.uniprot_isoform.isin(uni_iso)]
    # now merge with g2p data
    df_clf = df_clf.merge(
            df_g2p_f,
            how='left',
            on=['uniprot_isoform', 'aa_pos', 'aa_ref']
        )
    run_feature_boolean(df_clf, 'all_uniprot_sites', vsms)

Chromosome 1


/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_26621/4141627806.py:2: DtypeWarning: Columns (0: uniprot_isoform, 1: uniprot_id, 2: transcript_mane_select) have mixed types. Specify dtype option on import or set low_memory=False.
  df_l_c = pd.read_csv(


starting quantitative g2p features
Genes to complete: 1466


/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Genes to complete: 1466


...

starting quantitative g2p features
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
starting g2p binary features
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
starting secondary structure features
Genes to complete: 308
Genes to complete: 308
Genes to complete: 308
starting feature: G2P all sites
Genes to complete: 308
Chromosome X


/var/folders/13/9h9672rn4fq_js3d_7_vn7k40000gp/T/ipykernel_26621/4141627806.py:2: DtypeWarning: Columns (0: transcript_mane_select) have mixed types. Specify dtype option on import or set low_memory=False.
  df_l_c = pd.read_csv(


starting quantitative g2p features
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480


/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/enason/Documents/ai_agent_data/eval_tables/gcs_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, N

starting g2p binary features
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
starting secondary structure features
Genes to complete: 480
Genes to complete: 480
Genes to complete: 480
starting feature: G2P all sites
Genes to complete: 480
